# `TRG_EMP_load.txt` Conversion to Databricks Spark SQL

**Conversion Timestamp:** 2024-07-30 12:00:00

This notebook contains the Spark SQL conversion of the `TRG_EMP_load.txt` ODI session file. It performs a full load from the `EMPLOYEES` table into the `TRG_EMP` table.

In [ ]:
# SCEN_TASK_NO in {10}: Initialize ETL Parameters
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "-1", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "ETL Process ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")

## ETL Parameters

In [ ]:
%sql
-- Create temporary views for ETL parameters for easier SQL access
CREATE OR REPLACE TEMPORARY VIEW v_etl_job_type AS
SELECT '${ETL_JOB_TYPE}' AS etl_job_type;

CREATE OR REPLACE TEMPORARY VIEW v_datasource_num_id AS
SELECT CAST(${DATASOURCE_NUM_ID} AS BIGINT) AS datasource_num_id;

CREATE OR REPLACE TEMPORARY VIEW v_etl_proc_wid AS
SELECT CAST(${ETL_PROC_WID} AS BIGINT) AS etl_proc_wid;

CREATE OR REPLACE TEMPORARY VIEW v_odi_sess_no AS
SELECT '${ODI_SESS_NO}' AS odi_sess_no;

In [ ]:
display(spark.sql("SELECT * FROM v_etl_job_type"));
display(spark.sql("SELECT * FROM v_datasource_num_id"));
display(spark.sql("SELECT * FROM v_etl_proc_wid"));
display(spark.sql("SELECT * FROM v_odi_sess_no"));

## Data Loading

In [ ]:
%sql
-- SCEN_TASK_NO in {30}: Perform full load into TRG_EMP

-- For a full load, first delete all existing records from the target table.
DELETE FROM workspace.hr.trg_emp;

-- Insert all records from the source EMPLOYEES table into the target TRG_EMP table.
INSERT INTO workspace.hr.trg_emp
(
  employee_id,
  first_name,
  last_name,
  email,
  phone_number,
  hire_date,
  job_id,
  salary,
  commission_pct,
  manager_id,
  department_id
)
SELECT
  employees.employee_id,
  employees.first_name,
  employees.last_name,
  employees.email,
  employees.phone_number,
  employees.hire_date,
  employees.job_id,
  employees.salary,
  employees.commission_pct,
  employees.manager_id,
  employees.department_id
FROM
  workspace.hr.employees AS employees;

## Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_records_in_trg_emp
FROM workspace.hr.trg_emp;

## Conversion Notes

*   **Oracle Hints Removed:** The Oracle `/*+ APPEND PARALLEL */` hints were removed as they are not applicable to Databricks Delta Lake.
*   **Schema and Table Naming:** All Oracle schema (`HR`) and table names (`TRG_EMP`, `EMPLOYEES`) have been converted to lowercase and prefixed with `workspace.` (e.g., `workspace.hr.trg_emp`).
*   **Full Load Strategy:** The original ODI `INSERT` implies a full load or append. For a clean full load in Delta Lake, a `DELETE FROM` statement was added before the `INSERT` to ensure the target table is cleared before new data is inserted.
*   **SCEN_TASK_NO {10}, {20}**: These empty ODI steps are handled by the initial widget setup and general flow of the notebook.